# Lab 19.4: Is the Narrative Faithful to the Numbers?

Chapter 19 of *Explainable AI in Practice*. A linear model forecasts an invented importer's monthly sales from five factors. Its contributions to next January's forecast are exact (coefficient times the factor's deviation from its average). Narratives about those contributions, whether written by a person or a language model, are then checked automatically: are the named factors in the model, are directions and numbers right, is the largest factor named first?

Environment: numpy and pandas (`env/requirements.txt`). The optional last part needs only a browser and any chatbot you can use.

In [1]:
import re, numpy as np, pandas as pd
FEATURES = ["season", "fx_rate", "price_change", "promotion", "active_customers"]

rng = np.random.default_rng(7)
n = 60
month = np.arange(n) % 12
X = np.c_[np.cos(2 * np.pi * (month - 6) / 12),          # season (summer peak)
          np.cumsum(rng.normal(0.4, 1.0, n)) + 30,       # fx_rate: local currency per dollar
          rng.normal(0, 2, n),                           # price_change, percent
          (rng.random(n) < 0.25).astype(float),          # promotion month
          rng.normal(120, 8, n)]                         # active customers
beta = np.array([9.0, -1.6, -0.9, 6.0, 0.35])
y = 100 + X @ beta + rng.normal(0, 2.5, n)
A = np.c_[np.ones(n), X]
coef = np.linalg.lstsq(A, y, rcond=None)[0]
x_new = np.array([np.cos(2 * np.pi * (0 - 6) / 12), X[-1, 1] + 4.0, 0.5, 1.0, 121.0])   # next January
contrib = coef[1:] * (x_new - X.mean(0))
C = dict(zip(FEATURES, contrib))
pd.Series(C).round(2).sort_values()

fx_rate            -20.37
season              -9.09
price_change        -0.35
active_customers     0.42
promotion            5.31
dtype: float64

**Checkpoint 1:** fx_rate about -20.4, season about -9.1, promotion about +5.3, active_customers about +0.4, price_change about -0.35.

## 1. A small claim extractor and four checks

In [2]:
SYN = {"season": ["season", "seasonal"], "fx_rate": ["exchange rate", "currency", "dollar"],
       "price_change": ["price"], "promotion": ["promotion", "campaign", "discount"],
       "active_customers": ["customers", "customer base"]}
UP = ["raise", "raised", "lift", "lifted", "add", "added", "adding", "offset", "increase", "increased", "boost", "boosted"]
DOWN = ["lower", "lowered", "cut", "reduce", "reduced", "drag", "dragged", "decrease", "decreased", "fall", "dip"]

def extract_claims(text):
    out = []
    for sent in re.split(r"(?<=[.!?])\s+", text.strip()):
        low = sent.lower()
        feats = [f for f, keys in SYN.items() if any(k in low for k in keys)]
        d = "+" if any(re.search(rf"\b{u}\b", low) for u in UP) else ("-" if any(re.search(rf"\b{u}\b", low) for u in DOWN) else None)
        nums = [float(x) for x in re.findall(r"(\d+(?:\.\d+)?) units", low)]
        out.append(dict(sentence=sent, features=feats, direction=d, numbers=nums))
    return out

def check(text, tol=1.0):
    cl = extract_claims(text)
    named = [f for c in cl for f in c["features"]]
    unknown = [c["sentence"] for c in cl if not c["features"] and (c["direction"] or c["numbers"])]
    dirs = [(c["features"][0], c["direction"]) for c in cl if c["features"] and c["direction"]]
    nums = [(c["features"][0], c["numbers"][0]) for c in cl if c["features"] and c["numbers"]]
    top = max(FEATURES, key=lambda f: abs(C[f]))
    return {"factors in model": not unknown,
            "directions": all((C[f] > 0) == (d == "+") for f, d in dirs),
            "numbers": all(abs(abs(C[f]) - v) <= tol for f, v in nums),
            "largest named first": bool(named) and named[0] == top,
            "unmatched sentences": unknown}

## 2. Six prepared narratives

In [3]:
a = lambda f: f"{abs(C[f]):.1f}"
NARRATIVES = {
 "N1 faithful": f"Next January's forecast is lower mainly because the exchange rate rose, which lowered it by {a('fx_rate')} units. The seasonal winter dip lowered it by {a('season')} units. The planned promotion offset part of this, adding {a('promotion')} units.",
 "N2 wrong sign": f"Next January's forecast is lower mainly because the exchange rate rose, which lowered it by {a('fx_rate')} units. The planned promotion also dragged it down by {a('promotion')} units.",
 "N3 wrong order and number": f"The seasonal winter dip lowered the forecast by {abs(C['season']) + 3:.1f} units. The exchange rate lowered it by {a('fx_rate')} units.",
 "N4 invented cause": f"The forecast is lower mainly because the exchange rate rose, which lowered it by {a('fx_rate')} units. A national holiday in January will also cut sales.",
 "N5 paraphrase": f"A weaker local currency is the main reason the January figure is down, by about {a('fx_rate')} units. Winter is quieter too.",
 "N6 hedged and vague": "Several factors, including currency movements and the time of year, may be affecting next January's outlook.",
}
res = pd.DataFrame({k: check(v) for k, v in NARRATIVES.items()}).T
res

,factors in model,directions,numbers,largest named first,unmatched sentences
N1 faithful,True,True,True,True,[]
N2 wrong sign,True,False,True,True,[]
N3 wrong order and number,True,True,False,False,[]
N4 invented cause,False,True,True,True,[A national holiday in January will also cut s...
N5 paraphrase,True,True,True,True,[]
N6 hedged and vague,True,True,True,True,[]


**Checkpoint 2:** N1 passes every check; N2 fails directions; N3 fails numbers and order; N4 fails "factors in model". Look carefully at N5 and N6: what do the checks say, and is that right? (N5's "weaker local currency" is the fx_rate rising, since fx_rate is local currency per dollar; N6 makes no checkable claim at all.)

## 3. Your turn: narratives from a chatbot (optional, no code to write)

Give any chatbot the table below and ask it to explain next January's forecast to a small business owner in three sentences, using "units". Paste each answer between the quotes and rerun the cell. Record which errors the checker caught and which it missed.

In [4]:
table = "\n".join(f"{f}: {v:+.1f} units" for f, v in sorted(C.items(), key=lambda kv: kv[1]))
print(table)
MY_NARRATIVES = {
    "chatbot 1": "",   # paste here
    "chatbot 2": "",   # paste here
}
for name, text in MY_NARRATIVES.items():
    if text.strip():
        print(name, check(text))

fx_rate: -20.4 units
season: -9.1 units
price_change: -0.4 units
active_customers: +0.4 units
promotion: +5.3 units


## Questions

1. Which narrative errors are most dangerous for a business owner, and which do the four checks miss?
2. N5 is faithful but the extractor cannot fully see it. Would you rather accept some false alarms or some missed errors in a production checker? Why?
3. Write a rule (three or four lines) for when an LLM-written narrative may be shown to a user without human review.